# Mediterranean Sea Surface Temperature — Climatology, Anomalies & Extremes (1987–2025)

**Workflow**
1. Open the Mediterranean Sea Physics Reanalysis (`cmems_mod_med_phy-temp_my_4.2km_P1M-m`) — monthly means, lazy (dask).
2. Compute the **yearly mean SST** at every pixel and explore it with a year slider.
3. Compute the **long-term climatological mean** at every pixel (mean over 1987–2025).
4. Compute the **monthly SST anomaly** = monthly value − pixel climatological mean.
5. Visualise the monthly anomaly with an interactive time slider showing the current month.
6. At every pixel, find the **minimum** and **maximum** of the anomaly time-series.
7. Plot spatial maps of the min-anomaly and max-anomaly fields.
8. Export an **MP4 video** of the annual mean SST anomaly (1987–2025).

> **Credentials**: requires a `~/.netrc` entry for `auth.marine.copernicus.eu`.

In [ ]:
import cmocean
import copernicusmarine
import hvplot.xarray
import numpy as np
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# Mediterranean bounding box
MED_BBOX = dict(
    minimum_longitude=-6,
    maximum_longitude=42,
    minimum_latitude=30,
    maximum_latitude=47,
)

# The reanalysis starts January 1987; request through end of 2025
TIME_START = "1987-01-01"
TIME_END   = "2025-12-31"

## 1 — Open the SST dataset (lazy)

In [ ]:
%%time
ds = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_med_phy-temp_my_4.2km_P1M-m",
    start_datetime=TIME_START,
    end_datetime=TIME_END,
    **MED_BBOX,
)
ds

In [ ]:
# Surface SST: select depth nearest to 0 m — result is (time, latitude, longitude), still lazy
sst = ds["thetao"].sel(depth=0, method="nearest")
print(f"SST shape: {sst.dims} = {dict(zip(sst.dims, sst.shape))}")
sst

## 2 — Yearly mean SST with time slider

Resample monthly → annual, compute (≈ 39 frames × 380 × 1016 ≈ 60 MB), then display with a year scrubber.

In [ ]:
%%time
sst_yearly = sst.resample(time="YE").mean().compute()

# Replace the datetime coordinate with plain integer years for a cleaner slider
years = sst_yearly.time.dt.year.values
sst_yearly = (
    sst_yearly
    .assign_coords(year=("time", years))
    .swap_dims({"time": "year"})
    .drop_vars("time")
)
sst_yearly.attrs["long_name"] = "Annual Mean SST"
sst_yearly.attrs["units"] = "\u00b0C"
print(f"Yearly SST shape: {sst_yearly.dims} = {dict(zip(sst_yearly.dims, sst_yearly.shape))}")
sst_yearly

In [ ]:
sst_yearly.hvplot(
    x="longitude",
    y="latitude",
    groupby="year",
    rasterize=True,
    geo=True,
    cmap="turbo",
    clim=(float(sst_yearly.min()), float(sst_yearly.max())),
    clabel="SST (\u00b0C)",
    tiles="OSM",
    frame_width=800,
    frame_height=450,
    title="Mediterranean Annual Mean SST — use slider to step through years",
    fontscale=1.2,
    widget_type="scrubber",
    widget_location="bottom",
)

## 3 — Long-term climatological mean

Average every pixel over the full record (1987–2025). This is the reference field for anomaly computation.

In [ ]:
%%time
sst_clim = sst.mean(dim="time").compute()
sst_clim.attrs["long_name"] = f"Climatological Mean SST ({TIME_START[:4]}\u2013{TIME_END[:4]})"
sst_clim.attrs["units"] = "\u00b0C"
print(f"SST clim range: {float(sst_clim.min()):.1f} \u2013 {float(sst_clim.max()):.1f} \u00b0C")
sst_clim

In [ ]:
sst_clim.hvplot(
    x="longitude",
    y="latitude",
    rasterize=True,
    geo=True,
    cmap="turbo",
    clabel="SST (\u00b0C)",
    tiles="OSM",
    frame_width=800,
    frame_height=450,
    title=f"Mediterranean Climatological Mean SST ({TIME_START[:4]}\u2013{TIME_END[:4]})",
    fontscale=1.2,
)

## 4 — Monthly SST anomaly

Anomaly = monthly SST − pixel climatological mean.

The subtraction broadcasts `sst_clim` (2-D) over the time axis of `sst` (3-D). The result remains a lazy dask array — no data is loaded into RAM until we ask for an explicit reduction.

In [ ]:
sst_anom = sst - sst_clim
sst_anom.attrs["long_name"] = "SST Anomaly"
sst_anom.attrs["units"] = "\u00b0C"
print(f"Anomaly shape: {sst_anom.dims} = {dict(zip(sst_anom.dims, sst_anom.shape))}")
sst_anom

## 4b — Monthly SST anomaly map with time slider

In [ ]:
import pandas as pd
import panel as pn

pn.extension()

# Load the full monthly anomaly into memory — needed for a responsive time slider
sst_anom_loaded = sst_anom.compute()

# Symmetric color limits so blue/red are balanced around zero
anom_max = float(max(abs(sst_anom_loaded.min()), abs(sst_anom_loaded.max())))
times = sst_anom_loaded.time.values

# Integer-index Player widget (play/pause/step controls + scrubber)
player = pn.widgets.Player(
    start=0, end=len(times) - 1, value=0,
    loop_policy="loop", width=800, name="",
)

def plot_frame(idx):
    label = pd.Timestamp(times[idx]).strftime("%B %Y")
    return sst_anom_loaded.isel(time=idx).hvplot(
        x="longitude",
        y="latitude",
        rasterize=True,
        geo=True,
        cmap="RdBu_r",
        clim=(-anom_max, anom_max),
        clabel="SST Anomaly (\u00b0C)",
        tiles="OSM",
        frame_width=800,
        frame_height=450,
        title=f"Mediterranean Monthly SST Anomaly \u2014 {label}",
        fontscale=1.2,
    )

pn.Column(pn.bind(plot_frame, player), player)

## 5 — Pixel-wise minimum and maximum anomaly

Collapse the time dimension by taking `min` and `max`. Both reductions are fused into a single dask graph pass over the data.

In [ ]:
%%time
import dask

# Compute both reductions in one scheduler call to share the I/O
sst_anom_min, sst_anom_max = dask.compute(
    sst_anom.min(dim="time"),
    sst_anom.max(dim="time"),
)

sst_anom_min.attrs["long_name"] = "Minimum SST Anomaly"
sst_anom_min.attrs["units"] = "\u00b0C"
sst_anom_max.attrs["long_name"] = "Maximum SST Anomaly"
sst_anom_max.attrs["units"] = "\u00b0C"

print(f"Min anomaly range : {float(sst_anom_min.min()):.2f} \u2013 {float(sst_anom_min.max()):.2f} \u00b0C")
print(f"Max anomaly range : {float(sst_anom_max.min()):.2f} \u2013 {float(sst_anom_max.max()):.2f} \u00b0C")

## 6 — Spatial maps of min and max anomaly

A diverging colormap (`RdBu_r`) centred on zero makes cold/warm departures immediately readable.

In [ ]:
# Shared symmetric color limits so both maps are directly comparable
anom_abs_max = max(
    abs(float(sst_anom_min.min())),
    abs(float(sst_anom_max.max())),
)
clim_anom = (-anom_abs_max, anom_abs_max)
print(f"Symmetric color range: {clim_anom[0]:.2f} \u2013 {clim_anom[1]:.2f} \u00b0C")

In [ ]:
plot_min = sst_anom_min.hvplot(
    x="longitude",
    y="latitude",
    rasterize=True,
    geo=True,
    cmap="RdBu_r",
    clim=clim_anom,
    clabel="SST Anomaly (\u00b0C)",
    tiles="OSM",
    frame_width=800,
    frame_height=450,
    title=f"Mediterranean Minimum SST Anomaly ({TIME_START[:4]}\u2013{TIME_END[:4]})",
    fontscale=1.2,
)
plot_min

In [ ]:
plot_max = sst_anom_max.hvplot(
    x="longitude",
    y="latitude",
    rasterize=True,
    geo=True,
    cmap="RdBu_r",
    clim=clim_anom,
    clabel="SST Anomaly (\u00b0C)",
    tiles="OSM",
    frame_width=800,
    frame_height=450,
    title=f"Mediterranean Maximum SST Anomaly ({TIME_START[:4]}\u2013{TIME_END[:4]})",
    fontscale=1.2,
)
plot_max

## 7 — Side-by-side comparison

In [ ]:
plot_min_small = sst_anom_min.hvplot(
    x="longitude", y="latitude",
    rasterize=True, geo=True,
    cmap="RdBu_r", clim=clim_anom,
    clabel="SST Anomaly (\u00b0C)",
    tiles="OSM",
    frame_width=560, frame_height=350,
    title=f"Min Anomaly ({TIME_START[:4]}\u2013{TIME_END[:4]})",
    fontscale=1.1,
)

plot_max_small = sst_anom_max.hvplot(
    x="longitude", y="latitude",
    rasterize=True, geo=True,
    cmap="RdBu_r", clim=clim_anom,
    clabel="SST Anomaly (\u00b0C)",
    tiles="OSM",
    frame_width=560, frame_height=350,
    title=f"Max Anomaly ({TIME_START[:4]}\u2013{TIME_END[:4]})",
    fontscale=1.1,
)

(plot_min_small + plot_max_small).cols(2)

## 8 — Export annual mean SST anomaly animation as MP4

Resample the monthly SST anomaly to annual means, then render one frame per year (1987–2025) using matplotlib and save it as a downloadable MP4 video. A diverging colormap (`RdBu_r`) centred on zero makes cooler/warmer anomaly years immediately visible. Requires `ffmpeg` to be available on the system.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np

VIDEO_PATH = "med_annual_mean_sst_anomaly.mp4"

# Compute annual mean SST anomaly (year × lat × lon)
sst_anom_yearly = sst_anom.resample(time="YE").mean().compute()
anom_years = sst_anom_yearly.time.dt.year.values
sst_anom_yearly = (
    sst_anom_yearly
    .assign_coords(year=("time", anom_years))
    .swap_dims({"time": "year"})
    .drop_vars("time")
)
sst_anom_yearly.attrs["long_name"] = "Annual Mean SST Anomaly"
sst_anom_yearly.attrs["units"] = "\u00b0C"
print(f"Annual anomaly shape: {sst_anom_yearly.dims} = {dict(zip(sst_anom_yearly.dims, sst_anom_yearly.shape))}")

lons = sst_anom_yearly.longitude.values
lats = sst_anom_yearly.latitude.values
years_list = sst_anom_yearly.year.values

# Symmetric color limits across the entire record
vabs = float(max(
    abs(np.nanmin(sst_anom_yearly.values)),
    abs(np.nanmax(sst_anom_yearly.values)),
))
print(f"Symmetric color range: \u00b1{vabs:.2f} \u00b0C")

fig, ax = plt.subplots(figsize=(12, 6))
ax.set_facecolor("#d0e8f0")

data0 = sst_anom_yearly.sel(year=years_list[0]).values
img = ax.pcolormesh(
    lons, lats, data0,
    cmap="RdBu_r",
    vmin=-vabs, vmax=vabs,
    shading="auto",
)
cb = fig.colorbar(img, ax=ax, label="SST Anomaly (\u00b0C)", fraction=0.03, pad=0.02)
title = ax.set_title(
    f"Mediterranean Annual Mean SST Anomaly \u2014 {years_list[0]}",
    fontsize=14,
)
ax.set_xlabel("Longitude (\u00b0E)")
ax.set_ylabel("Latitude (\u00b0N)")
fig.tight_layout()

def update(frame_idx):
    yr = years_list[frame_idx]
    data = sst_anom_yearly.sel(year=yr).values
    img.set_array(data.ravel())
    title.set_text(f"Mediterranean Annual Mean SST Anomaly \u2014 {yr}")
    return img, title

ani = animation.FuncAnimation(
    fig,
    update,
    frames=len(years_list),
    interval=400,
    blit=False,
)

ani.save(VIDEO_PATH, writer="ffmpeg", fps=3, dpi=120)
plt.close(fig)
print(f"Video saved \u2192 {VIDEO_PATH}")

In [ ]:
from IPython.display import FileLink, display
display(FileLink(VIDEO_PATH, result_html_prefix="Download video: "))